# Smart Home Energy Forecasting
# Data preprocessing for time-series forecasting

This notebook follows the actual findings from the existing dataset inspection notebook. It checks the real columns and missing-value pattern before preparing the data for an LSTM model.


In [1]:
from pathlib import Path

import pandas as pd

# Load the dataset from the project data folder.
data_path = Path("../data/household_power_consumption.txt")
if not data_path.exists():
    data_path = Path("data/household_power_consumption.txt")

if not data_path.exists():
    raise FileNotFoundError("Dataset not found at ../data/household_power_consumption.txt or data/household_power_consumption.txt")

df = pd.read_csv(data_path, sep=';', na_values='?')
print("Dataset loaded successfully.")
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())


Dataset loaded successfully.
Shape: (2075259, 9)
Columns: ['Date', 'Time', 'Global_active_power', 'Global_reactive_power', 'Voltage', 'Global_intensity', 'Sub_metering_1', 'Sub_metering_2', 'Sub_metering_3']


### 1. Validate the actual dataset structure against the inspection notebook

The dataset inspection notebook showed the expected columns and the missing-value pattern. We verify the real file matches those findings before moving on.


In [2]:
expected_columns = ["Date", "Time", "Global_active_power", "Global_reactive_power", "Voltage", "Global_intensity", "Sub_metering_1", "Sub_metering_2", "Sub_metering_3"]

missing_columns = [col for col in expected_columns if col not in df.columns]
if missing_columns:
    raise ValueError(f"Missing expected columns: {missing_columns}")

print("Expected columns found.")
print("Missing values by column:")
print(df.isnull().sum())
print("\nGlobal_active_power missing count:", df["Global_active_power"].isnull().sum())


Expected columns found.
Missing values by column:
Date                         0
Time                         0
Global_active_power      25979
Global_reactive_power    25979
Voltage                  25979
Global_intensity         25979
Sub_metering_1           25979
Sub_metering_2           25979
Sub_metering_3           25979
dtype: int64

Global_active_power missing count: 25979


### 2. Convert Date and Time to a timestamp

The inspection notebook showed the file has Date and Time columns and that a combined timestamp is useful for time-order validation.


In [3]:
if "Date" in df.columns and "Time" in df.columns:
    df["timestamp"] = pd.to_datetime(df["Date"] + " " + df["Time"])
    print(df[["Date", "Time", "timestamp"]].head())
else:
    raise ValueError("The dataset does not contain the expected Date and Time columns.")


         Date      Time           timestamp
0  16/12/2006  17:24:00 2006-12-16 17:24:00
1  16/12/2006  17:25:00 2006-12-16 17:25:00
2  16/12/2006  17:26:00 2006-12-16 17:26:00
3  16/12/2006  17:27:00 2006-12-16 17:27:00
4  16/12/2006  17:28:00 2006-12-16 17:28:00


/var/folders/t1/pd3xbl0s3mv037q4k1zws1n40000gn/T/ipykernel_19822/952039611.py:2: UserWarning: Parsing dates in %d/%m/%Y %H:%M:%S format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df["timestamp"] = pd.to_datetime(df["Date"] + " " + df["Time"])


### 3. Handle missing values in the target column

The inspection notebook showed 25,979 missing values in Global_active_power. A simple beginner-friendly approach is to fill them before selecting the target series.


In [4]:
missing_before = df["Global_active_power"].isnull().sum()
print("Missing values before fill:", missing_before)

df["Global_active_power"] = df["Global_active_power"].ffill().bfill()
print("Missing values after fill:", df["Global_active_power"].isnull().sum())


Missing values before fill: 25979
Missing values after fill: 0


### 4. Select the target variable

We will predict Global_active_power, which is the main household active power measurement for this project.


In [5]:
from sklearn.preprocessing import MinMaxScaler

# Keep the target as a one-dimensional series with the original time order.
target = df["Global_active_power"].astype(float)
print("Target length:", len(target))
print("Target sample:", target.head().tolist())


Target length: 2075259
Target sample: [4.216, 5.36, 5.374, 5.388, 3.666]


### 5. Scale the target with MinMaxScaler

We scale the target values to the range [0, 1]. This keeps the LSTM training numerically stable and simple to understand.


In [6]:
scaler = MinMaxScaler()
scaled_target = scaler.fit_transform(target.to_numpy().reshape(-1, 1)).reshape(-1)
print("Scaled target sample:", scaled_target[:5].tolist())
print("Scaled target shape:", scaled_target.shape)


Scaled target sample: [0.37479630635524175, 0.47836320840123125, 0.4796306355241716, 0.48089806264711205, 0.32500452652543904]
Scaled target shape: (2075259,)


### 6. Create 60-minute input sequences and next-minute target values

Each input sequence uses the previous 60 minutes to predict the next minute. In this format, the input array has shape (samples, 60), and the target array has shape (samples, 1).


In [7]:
import numpy as np

sequence_length = 60

X = np.array([scaled_target[i : i + sequence_length] for i in range(len(scaled_target) - sequence_length)])
y = scaled_target[sequence_length:][:, None]

print("X shape:", X.shape)
print("y shape:", y.shape)
print("First X sequence sample:", X[0])
print("First y target sample:", y[0])


X shape: (2075199, 60)
y shape: (2075199, 1)
First X sequence sample: [0.37479631 0.47836321 0.47963064 0.48089806 0.32500453 0.31178707
 0.32826362 0.32808256 0.32518559 0.3246424  0.39579938 0.48307079
 0.46605106 0.4700344  0.36013036 0.29947492 0.28915445 0.30363933
 0.28879232 0.33061742 0.5267065  0.69074778 0.62918704 0.46152453
 0.39815318 0.28716277 0.28607641 0.28535216 0.28806808 0.28082564
 0.23936267 0.33333333 0.38620315 0.40159334 0.36049249 0.2169111
 0.2456998  0.23067174 0.24407025 0.33170378 0.43925403 0.54101032
 0.60438168 0.5792142  0.56418613 0.39724787 0.30056129 0.27285895
 0.33079848 0.2020641  0.20930654 0.40937896 0.4026797  0.37352888
 0.39797212 0.25131269 0.25819301 0.25927938 0.25873619 0.25801195]
First y target sample: [0.305631]


### 7. Split the sequences chronologically into training and testing sets

We must preserve the time order here. We do not shuffle the sequence data because that would break the forecasting setup.


In [8]:
train_size = int(len(X) * 0.8)

X_train = X[:train_size]
X_test = X[train_size:]
y_train = y[:train_size]
y_test = y[train_size:]

print("Train X shape:", X_train.shape)
print("Train y shape:", y_train.shape)
print("Test X shape:", X_test.shape)
print("Test y shape:", y_test.shape)
print("Chronological split complete.")


Train X shape: (1660159, 60)
Train y shape: (1660159, 1)
Test X shape: (415040, 60)
Test y shape: (415040, 1)
Chronological split complete.
